In [ ]:
import random
import json
from datetime import datetime, timedelta

import mysql.connector



TOTAL_TRANSACTIONS = 100000
FRAUD_PERCENTAGE = 0.02

random.seed(42)


conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mani@sql5825u",
    database="fraud_detection_final"
)

cursor = conn.cursor(dictionary=True)


cursor.execute("SELECT * FROM customers")

customers = cursor.fetchall()

print(f"Loaded {len(customers)} customers")


with open("model/customer_profiles.json", "r") as file:
    customer_profiles = json.load(file)

print(f"Loaded {len(customer_profiles)} customer profiles")

Loaded 1000 customers
Loaded 1000 customer profiles


In [ ]:


transaction_counts = {}

weights = []

for _ in customers:

    activity = random.choices(
        population=["Low", "Normal", "Active", "Heavy"],
        weights=[25, 45, 20, 10],
        k=1
    )[0]

    if activity == "Low":
        count = random.randint(20, 50)

    elif activity == "Normal":
        count = random.randint(60, 120)

    elif activity == "Active":
        count = random.randint(120, 250)

    else:
        count = random.randint(250, 600)

    weights.append(count)


scale = TOTAL_TRANSACTIONS / sum(weights)

scaled = [max(1, int(x * scale)) for x in weights]

difference = TOTAL_TRANSACTIONS - sum(scaled)

while difference != 0:

    i = random.randint(0, len(scaled) - 1)

    if difference > 0:
        scaled[i] += 1
        difference -= 1

    elif scaled[i] > 1:
        scaled[i] -= 1
        difference += 1

for customer, count in zip(customers, scaled):
    transaction_counts[customer["customer_id"]] = count

print("Transaction distribution created.")
print("Total =", sum(transaction_counts.values()))

Transaction distribution created.
Total = 100000


In [ ]:


cities = [
    "Chennai",
    "Coimbatore",
    "Madurai",
    "Trichy",
    "Salem",
    "Bangalore",
    "Hyderabad",
    "Mumbai",
    "Pune",
    "Delhi",
    "Kolkata",
    "Kochi"
]

devices = [
    "Samsung S24",
    "Samsung A55",
    "Samsung S23",
    "Samsung M35",
    "iPhone 15",
    "iPhone 16",
    "Redmi Note 13",
    "Redmi Note 14",
    "OnePlus 13",
    "Realme GT",
    "Vivo V50",
    "Oppo Reno 12"
]

payment_methods = [
    "UPI",
    "Debit Card",
    "Credit Card",
    "Net Banking"
]

In [ ]:


START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 6, 30)

def random_datetime(active_start, active_end):

    days = (END_DATE - START_DATE).days

    random_day = random.randint(0, days)

    hour = random.randint(active_start, active_end)

    minute = random.randint(0, 59)

    second = random.randint(0, 59)

    return START_DATE + timedelta(
        days=random_day,
        hours=hour,
        minutes=minute,
        seconds=second
    )

In [ ]:

def generate_normal_transaction(customer, profile):

    average = profile["average_amount"]

    variation = profile["variation"]

    active_start = profile["active_hours"]["start"]

    active_end = profile["active_hours"]["end"]

    

    amount = random.gauss(
        average,
        average * variation
    )

    amount = max(50, round(amount, 2))

    # Device

    if random.random() < 0.95:
        device = customer["usual_device"]
    else:
        device = random.choice(devices)

    # City

    if random.random() < 0.98:
        city = customer["home_city"]
    else:
        city = random.choice(cities)

    # Payment

    if random.random() < 0.90:
        payment = customer["usual_payment_method"]
    else:
        payment = random.choice(payment_methods)

    transaction_time = random_datetime(
        active_start,
        active_end
    )

    return {

        "transaction_time": transaction_time,

        "amount": amount,

        "payment_method": payment,

        "city": city,

        "device": device

    }

In [ ]:


def generate_fraud_transaction(customer, profile):

    tx = generate_normal_transaction(customer, profile)

    scenario = random.randint(1, 6)

    
    if scenario == 1:

        tx["amount"] *= random.uniform(5, 12)

    
    elif scenario == 2:

        tx["amount"] *= random.uniform(5, 10)

        tx["device"] = random.choice(
            [d for d in devices if d != customer["usual_device"]]
        )

    
    elif scenario == 3:

        tx["amount"] *= random.uniform(5, 10)

        tx["city"] = random.choice(
            [c for c in cities if c != customer["home_city"]]
        )

    
    elif scenario == 4:

        tx["amount"] *= random.uniform(5, 8)

        tx["payment_method"] = random.choice(
            [p for p in payment_methods if p != customer["usual_payment_method"]]
        )

    
    elif scenario == 5:

        tx["transaction_time"] = tx["transaction_time"].replace(
            hour=random.randint(1,4)
        )

    
    else:

        tx["device"] = random.choice(
            [d for d in devices if d != customer["usual_device"]]
        )

        tx["city"] = random.choice(
            [c for c in cities if c != customer["home_city"]]
        )

    tx["amount"] = round(tx["amount"],2)

    return tx

In [ ]:

insert_query = """
INSERT INTO transactions
(
    customer_id,
    transaction_time,
    amount,
    payment_method,
    city,
    device,
    fraud_label
)
VALUES (%s,%s,%s,%s,%s,%s,%s)
"""

transactions_to_insert = []

label_index = 0

print("\nGenerating Transactions...\n")

for customer in customers:

    customer_id = customer["customer_id"]

    profile = customer_profiles[str(customer_id)]

    total_tx = transaction_counts[customer_id]

    for _ in range(total_tx):

        fraud_label = labels[label_index]
        label_index += 1

        if fraud_label == 0:
            tx = generate_normal_transaction(customer, profile)
        else:
            tx = generate_fraud_transaction(customer, profile)

        transactions_to_insert.append(
            (
                customer_id,
                tx["transaction_time"],
                tx["amount"],
                tx["payment_method"],
                tx["city"],
                tx["device"],
                fraud_label
            )
        )

print(f"Generated {len(transactions_to_insert)} transactions.")


Generating Transactions...

Generated 100000 transactions.


In [ ]:


transactions_to_insert.sort(key=lambda x: x[1])

print("Transactions sorted chronologically.")

print("\nInserting into MySQL...")

BATCH_SIZE = 5000

for i in range(0, len(transactions_to_insert), BATCH_SIZE):

    batch = transactions_to_insert[i:i+BATCH_SIZE]

    cursor.executemany(insert_query, batch)

    conn.commit()

    print(f"Inserted {min(i+BATCH_SIZE, len(transactions_to_insert))} / {len(transactions_to_insert)}")

print("\n✅ Transaction generation completed successfully.")
cursor.close()
conn.close()

Transactions sorted chronologically.

Inserting into MySQL...
Inserted 5000 / 100000
Inserted 10000 / 100000
Inserted 15000 / 100000
Inserted 20000 / 100000
Inserted 25000 / 100000
Inserted 30000 / 100000
Inserted 35000 / 100000
Inserted 40000 / 100000
Inserted 45000 / 100000
Inserted 50000 / 100000
Inserted 55000 / 100000
Inserted 60000 / 100000
Inserted 65000 / 100000
Inserted 70000 / 100000
Inserted 75000 / 100000
Inserted 80000 / 100000
Inserted 85000 / 100000
Inserted 90000 / 100000
Inserted 95000 / 100000
Inserted 100000 / 100000

✅ Transaction generation completed successfully.
